# Group 2: Full Covariance Matrix of Stock Returns

The full covariance matrix of our 43 stocks' **time-weighted returns** on stagger 2 (the raw
returns, not the factor residuals). This is the matrix to send for the XRD covariance matrix.
All data comes from group0's `companies_and_staggered_returns.xlsx`.

- One row per 28-day return period on stagger 2 (80 periods).
- Sample covariance (divides by N - 1), in units of 28-day returns.
- DOCN and UPST list later than the other stocks, so they have fewer periods. Each pair of
  stocks uses the periods they both have (pairwise-complete), which is what `DataFrame.cov()`
  does by default.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = Path("../data/companies_and_staggered_returns.xlsx")
STAGGER = "stagger_2"
OUTPUT_FILE = Path("group2_return_covariance.xlsx")

## Load the stock returns

Same stock list and stagger 2 returns block as the factor-beta notebook.

In [2]:
companies = pd.read_excel(DATA_FILE, sheet_name="Companies")

# Keep only real stocks: one row per ticker, and no ETF / FX / SPY rows
stocks = (
    companies
    .dropna(subset=["sector"])
    .drop_duplicates(subset=["ticker"])
    .reset_index(drop=True)
)
tickers = stocks["ticker"].tolist()

returns = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=254, nrows=80, parse_dates=["date"]
).set_index("date")
stock_returns = returns[tickers].astype(float)

print(f"{len(tickers)} stocks, {len(stock_returns)} periods "
      f"({stock_returns.index.min():%Y-%m-%d} to {stock_returns.index.max():%Y-%m-%d})")
counts = stock_returns.count()
print("Stocks with fewer periods:", counts[counts < len(stock_returns)].to_dict())

43 stocks, 80 periods (2020-07-27 to 2026-08-17)
Stocks with fewer periods: {'DOCN': 70, 'UPST': 74}


## The covariance matrix

In [3]:
covariance = stock_returns.cov()          # pairwise-complete, N - 1
correlation = stock_returns.corr()

# Checks
assert covariance.shape == (len(tickers), len(tickers))
assert list(covariance.index) == tickers and list(covariance.columns) == tickers
assert np.allclose(covariance.values, covariance.values.T), "not symmetric"
assert np.allclose(np.diag(covariance), stock_returns.var()), "diagonal is not the variances"
assert not covariance.isna().any().any(), "missing values in the matrix"

eigenvalues = np.linalg.eigvalsh(covariance.values)
print(f"Symmetric: yes. Smallest eigenvalue: {eigenvalues.min():.3e} "
      f"({'positive definite' if eigenvalues.min() > 0 else 'NOT positive definite'})")
print(f"Volatility per 28-day period: {np.sqrt(np.diag(covariance)).min():.3f} "
      f"to {np.sqrt(np.diag(covariance)).max():.3f}")
covariance.round(5)

Symmetric: yes. Smallest eigenvalue: 1.398e-04 (positive definite)
Volatility per 28-day period: 0.043 to 0.302


,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,...,LLY,WELL,NOK,MSFT,GOOGL,COST,HD,KO,JNJ,BRK-B
VST,0.01762,0.00222,0.00051,0.00206,0.00309,0.00354,0.00306,0.00841,0.00397,0.00106,...,0.00099,0.00081,-0.00000,0.00189,0.00194,0.00161,0.00172,-0.00230,-0.00202,0.00019
VIST,0.00222,0.01660,-0.00003,0.00019,0.00123,-0.00116,0.00220,0.00330,0.00280,0.00518,...,0.00256,0.00063,0.00314,0.00074,0.00122,0.00173,-0.00162,0.00041,0.00007,0.00035
AEP,0.00051,-0.00003,0.00343,0.00253,-0.00114,-0.00136,-0.00113,0.00132,-0.00048,-0.00024,...,-0.00030,0.00204,0.00119,-0.00179,-0.00136,0.00046,0.00054,0.00079,0.00134,0.00067
ETR,0.00206,0.00019,0.00253,0.00318,-0.00024,-0.00083,-0.00065,0.00235,0.00060,0.00023,...,-0.00084,0.00182,0.00111,-0.00134,-0.00030,0.00094,0.00041,0.00029,0.00100,0.00056
DIS,0.00309,0.00123,-0.00114,-0.00024,0.00738,0.00453,0.00518,0.00308,0.00484,0.00260,...,0.00056,0.00017,0.00199,0.00408,0.00349,0.00112,0.00131,-0.00009,-0.00054,0.00054
CMG,0.00354,-0.00116,-0.00136,-0.00083,0.00453,0.01049,0.00286,0.00139,0.00053,-0.00197,...,0.00098,-0.00090,0.00022,0.00310,0.00148,0.00219,0.00216,0.00050,-0.00060,0.00029
AMZN,0.00306,0.00220,-0.00113,-0.00065,0.00518,0.00286,0.01002,0.00498,0.00647,0.00174,...,0.00187,0.00022,0.00467,0.00659,0.00638,0.00180,0.00100,-0.00063,-0.00228,0.00027
FIX,0.00841,0.00330,0.00132,0.00235,0.00308,0.00139,0.00498,0.01607,0.01247,0.00292,...,0.00164,0.00184,0.00561,0.00177,0.00461,0.00209,0.00232,-0.00031,0.00007,0.00114
STX,0.00397,0.00280,-0.00048,0.00060,0.00484,0.00053,0.00647,0.01247,0.02759,0.00519,...,0.00374,0.00187,0.01164,0.00301,0.00951,0.00207,0.00220,0.00171,0.00099,0.00052
VLO,0.00106,0.00518,-0.00024,0.00023,0.00260,-0.00197,0.00174,0.00292,0.00519,0.01075,...,0.00125,0.00116,0.00115,0.00128,0.00159,0.00045,0.00015,0.00035,0.00096,0.00129


## Save to Excel

Sheets: **Covariance** (the matrix, diagonal in bold), **Correlation**, **Returns** (the
inputs, so the matrix can be checked), and **Notes**.

In [4]:
from openpyxl.styles import Font

notes = pd.DataFrame({
    "Item": ["Group", "Stagger", "Returns", "Periods", "Stocks", "Estimator",
             "Missing data", "Units", "Smallest eigenvalue"],
    "Value": ["Group 2", STAGGER, "Time-weighted stock returns (raw, not residuals)",
              f"{len(stock_returns)} ({stock_returns.index.min():%Y-%m-%d} to "
              f"{stock_returns.index.max():%Y-%m-%d})",
              len(tickers), "Sample covariance (N - 1)",
              "Pairwise-complete: " + ", ".join(f"{t} has {n} periods"
                                               for t, n in counts[counts < len(stock_returns)].items()),
              "Per 28-day period (not annualised)", f"{eigenvalues.min():.3e}"],
})

returns_out = stock_returns.copy()
returns_out.index = returns_out.index.strftime("%Y-%m-%d")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for name, matrix in [("Covariance", covariance), ("Correlation", correlation)]:
        matrix.rename_axis("ticker").to_excel(writer, sheet_name=name)
        ws = writer.sheets[name]
        for i, row in enumerate(ws.iter_rows(min_row=2, min_col=2)):
            for j, c in enumerate(row):
                c.number_format = "0.000000" if name == "Covariance" else "0.000"
                if i == j:
                    c.font = Font(bold=True)
        ws.freeze_panes = "B2"
    returns_out.rename_axis("date").to_excel(writer, sheet_name="Returns")
    for row in writer.sheets["Returns"].iter_rows(min_row=2, min_col=2):
        for c in row:
            c.number_format = "0.0000"
    writer.sheets["Returns"].freeze_panes = "B2"
    notes.to_excel(writer, sheet_name="Notes", index=False)
    writer.sheets["Notes"].column_dimensions["A"].width = 22
    writer.sheets["Notes"].column_dimensions["B"].width = 60

print(f"saved {OUTPUT_FILE.resolve().name}")

saved group2_return_covariance.xlsx
